# Demand Destruction Analysis
**Oil Super Spike Analyzer — Historical Calibration Notebook**

Implements four methods from `demand_destruction_tracking.md`:
1. **Price Elasticity (OLS)** — ε from FRED Brent + IEA/EIA demand
2. **Counterfactual Trend Deviation** — actual vs. pre-war trend
3. **Supply vs. Demand Shock Decomposition** — inventory identity
4. **Regime Classification** — label current conditions

All data pulled from the live project DB + FRED API (keys in `.env`).

In [1]:
import sys, os
sys.path.insert(0, r'd:\oil')   # project root on PATH

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from fredapi import Fred
from sqlalchemy import text

from config import FRED_API_KEY, DB_PATH
from db.database import get_session, SupplyDemand, FuturesCurve

fred  = Fred(api_key=FRED_API_KEY)
session = get_session()

# ── Plotly dark theme ────────────────────────────────────────────────────────
DARK = dict(
    template='plotly_dark',
    paper_bgcolor='#12121f',
    plot_bgcolor='#12121f',
    font=dict(color='#D0D0D0', family='Segoe UI, Arial'),
)
AXIS = dict(gridcolor='#2c2c42', zerolinecolor='#4a4a62')

print('Setup complete. DB:', DB_PATH)

Setup complete. DB: D:\oil\db\oil_data.db


## 1  Load Price + Demand Data
- **Brent spot** from FRED `DCOILBRENTEU` (daily → monthly avg)
- **WTI spot** from FRED `DCOILWTICO`
- **EIA international demand** from `supply_demand` DB table (annual, 25 countries)

In [2]:
# ── 1a. Brent & WTI from FRED ────────────────────────────────────────────────
brent_raw = fred.get_series('DCOILBRENTEU', observation_start='2005-01-01')
wti_raw   = fred.get_series('DCOILWTICO',   observation_start='2005-01-01')

brent_m = brent_raw.resample('ME').mean().rename('brent')
wti_m   = wti_raw.resample('ME').mean().rename('wti')
price   = pd.concat([brent_m, wti_m], axis=1).dropna(how='all')
price.index = price.index.to_period('M').to_timestamp('M')

print(f'Price series: {price.index[0].date()} → {price.index[-1].date()}, {len(price)} months')
price.tail()

Price series: 2005-01-31 → 2026-05-31, 257 months


,brent,wti
2026-01-31,66.602381,60.037000
2026-02-28,70.887000,64.508421
2026-03-31,103.134545,91.383636
2026-04-30,117.287500,100.316667
2026-05-31,118.260000,107.570000


In [3]:

# ── 1b. EIA annual global demand from DB ─────────────────────────────────────
rows = session.query(SupplyDemand).filter(
    SupplyDemand.flow == 'demand',
    SupplyDemand.product == 'total',
).all()

demand_db = pd.DataFrame([
    {'date': r.date, 'country': r.country, 'kbd': r.value_kbd}
    for r in rows
])
demand_db['year'] = pd.to_datetime(demand_db['date']).dt.year

# Drop incomplete years (< 20 countries reporting — partial data)
country_counts = demand_db.groupby('year')['country'].nunique()
complete_years = country_counts[country_counts >= 20].index
demand_db = demand_db[demand_db['year'].isin(complete_years)]

# Global total — sum across all 25 tracked countries
global_demand = (
    demand_db.groupby('year')['kbd'].sum()
    .rename('global_kbd')
    .reset_index()
)

print(f'Demand rows: {len(demand_db)} | Countries: {demand_db["country"].nunique()} | Years: {sorted(demand_db["year"].unique())}')
global_demand


Demand rows: 250 | Countries: 25 | Years: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]


,year,global_kbd
0,2015,73147.770290
1,2016,74089.616058
2,2017,75907.952792
3,2018,76228.750659
4,2019,76947.871164
5,2020,70123.997152
6,2021,74705.810681
7,2022,76346.723116
8,2023,77608.984550
9,2024,78716.101568


In [4]:
# ── 1c. Brent annual average — align with demand years ───────────────────────
price['year'] = price.index.year
brent_annual = price.groupby('year')['brent'].mean().reset_index()

combined = global_demand.merge(brent_annual, on='year').dropna()
combined['log_demand'] = np.log(combined['global_kbd'])
combined['log_price']  = np.log(combined['brent'])
combined['demand_yoy'] = combined['global_kbd'].pct_change() * 100
combined['price_yoy']  = combined['brent'].pct_change() * 100

# Tag historical episodes
episodes = {
    2008: ('2008 Financial Crisis', '#EF5350'),
    2009: ('2008 Financial Crisis', '#EF5350'),
    2020: ('COVID-19 Lockdowns',    '#FF9800'),
    2022: ('Ukraine War Spike',     '#7E57C2'),
}
combined['episode'] = combined['year'].map(lambda y: episodes.get(y, (str(y), '#4FC3F7'))[0])
combined['ecolor']  = combined['year'].map(lambda y: episodes.get(y, (str(y), '#4FC3F7'))[1])

print(combined[['year','global_kbd','brent','demand_yoy','price_yoy']].round(1).to_string(index=False))

 year  global_kbd  brent  demand_yoy  price_yoy
 2015     73147.8   52.4         NaN        NaN
 2016     74089.6   43.5         1.3      -16.8
 2017     75908.0   54.2         2.5       24.6
 2018     76228.8   71.1         0.4       31.0
 2019     76947.9   64.4         0.9       -9.4
 2020     70124.0   41.8        -8.9      -35.1
 2021     74705.8   70.7         6.5       69.3
 2022     76346.7  100.8         2.2       42.6
 2023     77609.0   82.5         1.7      -18.2
 2024     78716.1   80.5         1.4       -2.4


## 2  Method 1 — Price Elasticity of Demand (OLS)

$$\ln\!\left(\tfrac{D_t}{D_{t-1}}\right) = \alpha + \varepsilon \cdot \ln\!\left(\tfrac{P_t}{P_{t-1}}\right) + u_t$$

Expected short-run $\varepsilon$: **−0.05 to −0.08** (IEA consensus).

In [5]:

ols_df = combined.dropna(subset=['demand_yoy','price_yoy']).copy()

# OLS: log-log changes
log_dD = np.log(ols_df['global_kbd'] / ols_df['global_kbd'].shift(1)).dropna()
log_dP = np.log(ols_df['brent']      / ols_df['brent'].shift(1)).dropna()
idx    = log_dD.index.intersection(log_dP.index)
X = log_dP.loc[idx].values.reshape(-1,1)
y = log_dD.loc[idx].values

slope, intercept, r, p, se = stats.linregress(X.flatten(), y)

# ── NOTE on sign ─────────────────────────────────────────────────────────────
# With only 2015-2024 our window is dominated by two demand-driven shocks:
#   COVID-19 (2020): price -35%, demand -8.9% → strong positive co-movement
#   Recovery (2021): price +69%, demand +6.5% → same direction
# These income/activity shocks dominate, producing a positive OLS slope.
# True supply-side demand destruction elasticity (negative) requires isolating
# *supply-driven* price spikes (e.g. 1973, 1979, 2008 supply crunch).
# We therefore use the IEA consensus short-run ε = -0.06 for forward scenarios,
# and treat the OLS result as a "demand sensitivity to activity cycles."
# ─────────────────────────────────────────────────────────────────────────────
IEA_ELASTICITY = -0.06   # IEA consensus short-run price elasticity of demand

print(f'┌─ OLS Regression Result (activity-cycle sensitivity) ───────')
print(f'│  β (fitted)      = {slope:.4f}  (SE={se:.4f}, p={p:.3f})')
print(f'│  R²              = {r**2:.3f}')
print(f'│  Sign: POSITIVE — dataset dominated by income shocks (COVID/recovery)')
print(f'│  → OLS captures demand-activity correlation, not price elasticity')
print(f'│')
print(f'│  IEA consensus short-run ε = {IEA_ELASTICITY:.2f} (used in scenarios below)')
print(f'│  At $150/bbl (+50% from $100): IEA ε implies demand Δ = {IEA_ELASTICITY*50:.1f}%')

GLOBAL_BASE_KBD = combined['global_kbd'].iloc[-1]
delta_pct = IEA_ELASTICITY * 50
delta_kbd  = GLOBAL_BASE_KBD * delta_pct / 100
print(f'│  = {delta_kbd:+,.0f} kbd  ({delta_kbd/1000:+.2f} Mb/d) vs {GLOBAL_BASE_KBD:,.0f} kbd baseline')
print(f'└────────────────────────────────────────────────────────────')


┌─ OLS Regression Result (activity-cycle sensitivity) ───────
│  β (fitted)      = 0.1098  (SE=0.0348, p=0.020)
│  R²              = 0.624
│  Sign: POSITIVE — dataset dominated by income shocks (COVID/recovery)
│  → OLS captures demand-activity correlation, not price elasticity
│
│  IEA consensus short-run ε = -0.06 (used in scenarios below)
│  At $150/bbl (+50% from $100): IEA ε implies demand Δ = -3.0%
│  = -2,361 kbd  (-2.36 Mb/d) vs 78,716 kbd baseline
└────────────────────────────────────────────────────────────


In [6]:

# ── Scatter: Δlog(demand) vs Δlog(price) with both fitted lines ───────────────
x_fit = np.linspace(X.min()-0.02, X.max()+0.02, 100)
y_fit_ols = intercept + slope * x_fit         # OLS: activity-cycle sensitivity
y_fit_iea = 0 + IEA_ELASTICITY * x_fit        # IEA: supply-side elasticity

fig = go.Figure()
for _, row in ols_df.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['price_yoy']], y=[row['demand_yoy']],
        mode='markers+text',
        text=[str(int(row['year']))],
        textposition='top center',
        marker=dict(color=row['ecolor'], size=10, line=dict(color='white', width=0.5)),
        showlegend=False,
        hovertemplate=f"<b>{int(row['year'])}</b><br>Price YoY: %{{x:.1f}}%<br>Demand YoY: %{{y:.1f}}%<extra></extra>",
    ))

fig.add_trace(go.Scatter(
    x=x_fit*100, y=y_fit_ols*100,
    mode='lines',
    name=f'OLS fit β={slope:.3f} R²={r**2:.2f} (activity cycle)',
    line=dict(color='#4FC3F7', width=2, dash='dash'),
))
fig.add_trace(go.Scatter(
    x=x_fit*100, y=y_fit_iea*100,
    mode='lines',
    name=f'IEA consensus ε=−0.06 (supply-side scenarios)',
    line=dict(color='#EF5350', width=2, dash='dot'),
))

fig.update_layout(
    **DARK,
    title='Price Elasticity — OLS (activity cycle) vs. IEA Consensus (supply shock)',
    xaxis=dict(**AXIS, title='Brent Price YoY %'),
    yaxis=dict(**AXIS, title='Global Demand YoY %'),
    height=480,
    annotations=[dict(
        text='OLS positive slope = income shock dominance<br>(COVID fell together; recovery rose together)',
        xref='paper', yref='paper', x=0.02, y=0.97,
        showarrow=False, font=dict(size=10, color='#AAA'),
        bgcolor='rgba(18,18,31,0.8)', bordercolor='#555',
    )],
)
fig.add_hline(y=0, line=dict(color='#555', dash='dot'))
fig.add_vline(x=0, line=dict(color='#555', dash='dot'))
fig.show()


## 3  Method 2 — Counterfactual Trend Deviation

Fit a linear trend through 2015–2019 (pre-COVID, pre-war), extrapolate to 2026, and measure the gap to actual demand.

$$\text{Destruction}_t = \hat{D}_t^{\text{trend}} - D_t^{\text{actual}}$$

In [7]:
# Fit trend on 2015–2019 only
trend_base = combined[(combined['year'] >= 2015) & (combined['year'] <= 2019)].copy()
slope_t, intercept_t, *_ = stats.linregress(trend_base['year'], trend_base['global_kbd'])

# Extrapolate 2015 → 2026
years_ext = np.arange(2015, combined['year'].max() + 2)
trend_line = pd.DataFrame({'year': years_ext, 'trend_kbd': intercept_t + slope_t * years_ext})

cf = combined.merge(trend_line, on='year', how='left')
cf['destruction_kbd'] = cf['trend_kbd'] - cf['global_kbd']   # positive = destroyed

cumulative_loss = cf[cf['destruction_kbd'] > 0]['destruction_kbd'].sum()
print(f'Trend: +{slope_t:,.0f} kbd/year from {intercept_t:,.0f} kbd base')
print(f'Cumulative demand below trend (positive episodes): {cumulative_loss:,.0f} kbd·years')
print()
print(cf[['year','global_kbd','trend_kbd','destruction_kbd']].round(0).to_string(index=False))

Trend: +974 kbd/year from -1,889,160 kbd base
Cumulative demand below trend (positive episodes): 23,812 kbd·years

 year  global_kbd  trend_kbd  destruction_kbd
 2015     73148.0    73317.0            169.0
 2016     74090.0    74290.0            201.0
 2017     75908.0    75264.0           -644.0
 2018     76229.0    76238.0             10.0
 2019     76948.0    77212.0            264.0
 2020     70124.0    78186.0           8062.0
 2021     74706.0    79160.0           4454.0
 2022     76347.0    80134.0           3787.0
 2023     77609.0    81108.0           3499.0
 2024     78716.0    82082.0           3366.0


In [8]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.65, 0.35],
                    vertical_spacing=0.06,
                    subplot_titles=('Global Demand vs. Pre-War Trend', 'Demand Below Trend (Destruction Gap, kbd)'))

# ── Row 1: actual vs trend ────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=cf['year'], y=cf['global_kbd'],
    name='Actual Demand (EIA)', mode='lines+markers',
    line=dict(color='#4FC3F7', width=2.5),
    marker=dict(size=7),
    hovertemplate='%{x}: <b>%{y:,.0f} kbd</b><extra>Actual</extra>',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=trend_line['year'], y=trend_line['trend_kbd'],
    name='2015–2019 Trend (extrapolated)', mode='lines',
    line=dict(color='#FFB300', width=2, dash='dash'),
    hovertemplate='%{x}: <b>%{y:,.0f} kbd</b><extra>Trend</extra>',
), row=1, col=1)

# Shade gap where actual < trend
gap_years = cf[cf['destruction_kbd'] > 0]
fig.add_trace(go.Scatter(
    x=pd.concat([gap_years['year'], gap_years['year'][::-1]]),
    y=pd.concat([gap_years['trend_kbd'], gap_years['global_kbd'][::-1]]),
    fill='toself', fillcolor='rgba(239,83,80,0.18)',
    line=dict(color='rgba(0,0,0,0)'),
    name='Destruction zone', showlegend=True,
    hoverinfo='skip',
), row=1, col=1)

# Annotate episodes
for yr, (label, color) in episodes.items():
    row_ = cf[cf['year']==yr]
    if row_.empty: continue
    fig.add_annotation(x=yr, y=row_['global_kbd'].values[0],
                       text=label, showarrow=True, arrowhead=2,
                       arrowcolor=color, font=dict(color=color, size=9),
                       bgcolor='rgba(18,18,31,0.8)', row=1, col=1)

# ── Row 2: destruction gap bar ────────────────────────────────────────────────
bar_colors = ['#EF5350' if v > 0 else '#4CAF50' for v in cf['destruction_kbd']]
fig.add_trace(go.Bar(
    x=cf['year'], y=cf['destruction_kbd'],
    marker_color=bar_colors,
    name='Gap (kbd)', showlegend=False,
    hovertemplate='%{x}: <b>%{y:+,.0f} kbd</b><extra>Gap vs trend</extra>',
), row=2, col=1)
fig.add_hline(y=0, line=dict(color='#555', dash='dot'), row=2, col=1)

fig.update_layout(**DARK, height=620, title='Method 2 — Counterfactual Trend Deviation')
fig.update_xaxes(**AXIS)
fig.update_yaxes(**AXIS)
fig.show()

## 4  Method 3 — Supply vs. Demand Shock Decomposition

Uses the inventory identity to infer implied demand vs. supply separately:

$$\text{Implied Demand}_t = \text{Supply}_t - \Delta\text{Inventories}_t$$

A price rise with **falling inventories** = supply shock (bullish).  
A price fall with **rising inventories** = demand destruction (bearish).

In [12]:

# ── Supply from DB — average kbd across months to get annual rate ─────────────
supply_rows = session.query(SupplyDemand).filter(
    SupplyDemand.flow == 'supply',
    SupplyDemand.product == 'total',
).all()
supply_db = pd.DataFrame([{'date': r.date, 'country': r.country, 'kbd': r.value_kbd} for r in supply_rows])
supply_db['year'] = pd.to_datetime(supply_db['date']).dt.year

# Average monthly values per country × year, then sum countries → annual rate (kbd)
supply_annual_by_country = supply_db.groupby(['year','country'])['kbd'].mean()
global_supply = supply_annual_by_country.groupby('year').sum().rename('supply_kbd').reset_index()

# Keep only years with demand data (2015–2024)
global_supply = global_supply[global_supply['year'].isin(complete_years)]

print('Global supply (kbd, annual avg):')
print(global_supply.to_string(index=False))
print()

# ── US crude inventory from FRED (WCESTUS1 = weekly stocks in Mbbl) ───────────
try:
    inv_raw = fred.get_series('WCESTUS1', observation_start='2010-01-01')
    inv_annual = inv_raw.resample('YE').mean().rename('us_inv_mbbl')
    inv_annual.index = inv_annual.index.year
    inv_annual = inv_annual.reset_index().rename(columns={'index': 'year'})
    print(f'US inventory series: {len(inv_annual)} annual points')
except Exception as e:
    print(f'FRED inventory fetch failed ({e}), estimating from supply-demand balance')
    inv_annual = None

# ── Merge supply + demand + price ────────────────────────────────────────────
decomp = global_supply.merge(global_demand, on='year').merge(brent_annual, on='year')
decomp['balance_kbd']  = decomp['supply_kbd'] - decomp['global_kbd']
decomp['balance_mbd']  = decomp['balance_kbd'] / 1000
decomp['supply_yoy']   = decomp['supply_kbd'].pct_change() * 100
decomp['demand_yoy_d'] = decomp['global_kbd'].pct_change() * 100
decomp['price_yoy']    = decomp['brent'].pct_change() * 100

def classify(row):
    if row['balance_kbd'] > 500 and row['price_yoy'] < 0:
        return 'Demand Destruction'
    elif row['balance_kbd'] < -500 and row['price_yoy'] > 0:
        return 'Supply Shock'
    elif row['balance_kbd'] > 500 and row['price_yoy'] > 0:
        return 'Supply Surge (bearish lag)'
    else:
        return 'Balanced'

decomp['regime'] = decomp.apply(classify, axis=1)
decomp[['year','supply_kbd','global_kbd','balance_mbd','brent','price_yoy','regime']].round(2)


Global supply (kbd, annual avg):
 year   supply_kbd
 2015 78193.215404
 2016 79142.216228
 2017 80516.546869
 2018 83131.696812
 2019 83076.325583
 2020 77750.622183
 2021 79617.801615
 2022 84191.895821
 2023 86139.537026
 2024 86795.723105

FRED inventory fetch failed (Bad Request.  The series does not exist.), estimating from supply-demand balance


,year,supply_kbd,global_kbd,balance_mbd,brent,price_yoy,regime
0,2015,78193.22,73147.77,5.05,52.35,NaN,Balanced
1,2016,79142.22,74089.62,5.05,43.55,-16.81,Demand Destruction
2,2017,80516.55,75907.95,4.61,54.25,24.57,Supply Surge (bearish lag)
3,2018,83131.70,76228.75,6.90,71.06,30.99,Supply Surge (bearish lag)
4,2019,83076.33,76947.87,6.13,64.35,-9.44,Demand Destruction
5,2020,77750.62,70124.00,7.63,41.76,-35.11,Demand Destruction
6,2021,79617.80,74705.81,4.91,70.68,69.25,Supply Surge (bearish lag)
7,2022,84191.90,76346.72,7.85,100.78,42.59,Supply Surge (bearish lag)
8,2023,86139.54,77608.98,8.53,82.47,-18.17,Demand Destruction
9,2024,86795.72,78716.10,8.08,80.53,-2.36,Demand Destruction


In [13]:
regime_colors = {
    'Demand Destruction':       '#EF5350',
    'Supply Shock':             '#7E57C2',
    'Supply Surge (bearish lag)':'#FFB300',
    'Balanced':                 '#4FC3F7',
}

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Supply–Demand Balance (Mb/d)', 'Shock Decomposition: Balance vs. Price YoY'),
                    horizontal_spacing=0.1)

# ── Left: waterfall-style balance over time ───────────────────────────────────
fig.add_trace(go.Bar(
    x=decomp['year'], y=decomp['balance_mbd'],
    marker_color=['#EF5350' if v < 0 else '#4CAF50' for v in decomp['balance_mbd']],
    name='Balance (Mb/d)',
    hovertemplate='%{x}: <b>%{y:+.2f} Mb/d</b><extra>Balance</extra>',
), row=1, col=1)
fig.add_hline(y=0, line=dict(color='#aaa', dash='dot'), row=1, col=1)

# Overlay Brent on secondary axis
fig.add_trace(go.Scatter(
    x=decomp['year'], y=decomp['brent'],
    name='Brent ($/bbl)', mode='lines',
    line=dict(color='#FFB300', width=2),
    yaxis='y3',
    hovertemplate='%{x}: <b>$%{y:.0f}</b><extra>Brent</extra>',
), row=1, col=1)

# ── Right: scatter — balance vs price_yoy coloured by regime ─────────────────
for regime, grp in decomp.groupby('regime'):
    fig.add_trace(go.Scatter(
        x=grp['price_yoy'], y=grp['balance_mbd'],
        mode='markers+text',
        text=grp['year'].astype(str),
        textposition='top center',
        name=regime,
        marker=dict(color=regime_colors.get(regime,'#fff'), size=11,
                    line=dict(color='white', width=0.5)),
        hovertemplate='<b>%{text}</b><br>Price YoY: %{x:.1f}%<br>Balance: %{y:+.2f} Mb/d<extra></extra>',
    ), row=1, col=2)

fig.add_hline(y=0,  line=dict(color='#555', dash='dot'), row=1, col=2)
fig.add_vline(x=0,  line=dict(color='#555', dash='dot'), row=1, col=2)

fig.update_layout(
    **DARK, height=480,
    title='Method 3 — Supply vs. Demand Shock Decomposition',
    yaxis3=dict(overlaying='y', side='right', title='Brent $/bbl',
                tickfont=dict(color='#FFB300'), showgrid=False),
)
fig.update_xaxes(**AXIS)
fig.update_yaxes(**AXIS)
fig.show()

## 5  Method 4 — Regime Classification (KNN)

Train on the 5 labelled historical episodes; classify **current conditions** using:
- Brent 3-month % change
- Supply–demand balance (Mb/d)
- Price level vs. 3-year avg

In [14]:
# ── Training labels (manually from literature) ───────────────────────────────
# Mild=0, Moderate=1, Severe=2, Recovery=3
REGIME_LABELS = {
    2008: ('Severe',   2),   # GFC
    2009: ('Recovery', 3),
    2014: ('Moderate', 1),   # supply glut begins
    2016: ('Recovery', 3),
    2020: ('Severe',   2),   # COVID
    2021: ('Recovery', 3),
    2022: ('Mild',     0),   # war spike — demand held
    2023: ('Moderate', 1),   # recession fears
}
REGIME_COLORS = {'Severe': '#EF5350', 'Moderate': '#FF9800', 'Mild': '#FFD700', 'Recovery': '#4CAF50'}

# Build feature set from decomp
train_rows = []
for yr, (label, code) in REGIME_LABELS.items():
    row = decomp[decomp['year'] == yr]
    if row.empty: continue
    train_rows.append({
        'year': yr, 'label': label, 'code': code,
        'price_yoy':   row['price_yoy'].values[0],
        'balance_mbd': row['balance_mbd'].values[0],
        'brent':       row['brent'].values[0],
    })

train_df = pd.DataFrame(train_rows)

# Add price vs. 3-year rolling avg
brent_3yr = brent_annual.set_index('year')['brent'].rolling(3).mean()
train_df['price_vs_3yr'] = train_df.apply(
    lambda r: (r['brent'] - brent_3yr.get(r['year'], r['brent'])) / (brent_3yr.get(r['year'], r['brent']) + 1e-9) * 100, axis=1
)

features = ['price_yoy', 'balance_mbd', 'price_vs_3yr']
X_train = train_df[features].values
y_train = train_df['code'].values

scaler = StandardScaler().fit(X_train)
knn = KNeighborsClassifier(n_neighbors=3, weights='distance')
knn.fit(scaler.transform(X_train), y_train)

# ── Classify every year in decomp ────────────────────────────────────────────
decomp['price_vs_3yr'] = decomp.apply(
    lambda r: (r['brent'] - brent_3yr.get(r['year'], r['brent'])) / (brent_3yr.get(r['year'], r['brent']) + 1e-9) * 100, axis=1
)

LABEL_MAP = {0:'Mild', 1:'Moderate', 2:'Severe', 3:'Recovery'}
X_all = decomp[features].fillna(0).values
decomp['knn_code']  = knn.predict(scaler.transform(X_all))
decomp['knn_label'] = decomp['knn_code'].map(LABEL_MAP)
decomp['knn_prob']  = knn.predict_proba(scaler.transform(X_all)).max(axis=1)

# ── Current conditions (most recent year in decomp) ──────────────────────────
current = decomp.iloc[-1]
print(f'\n── Current Regime Assessment ({int(current["year"])}) ──────────────────────')
print(f'  Brent YoY:        {current["price_yoy"]:+.1f}%')
print(f'  S-D Balance:      {current["balance_mbd"]:+.2f} Mb/d')
print(f'  Price vs 3yr avg: {current["price_vs_3yr"]:+.1f}%')
print(f'  KNN Regime:       {current["knn_label"]}  (confidence {current["knn_prob"]*100:.0f}%)')
print()
decomp[['year','knn_label','knn_prob','brent','balance_mbd','price_yoy']].tail(8).round(2)


── Current Regime Assessment (2024) ──────────────────────
  Brent YoY:        -2.4%
  S-D Balance:      +8.08 Mb/d
  Price vs 3yr avg: -8.4%
  KNN Regime:       Moderate  (confidence 58%)



,year,knn_label,knn_prob,brent,balance_mbd,price_yoy
2,2017,Recovery,0.78,54.25,4.61,24.57
3,2018,Mild,0.50,71.06,6.90,30.99
4,2019,Recovery,0.36,64.35,6.13,-9.44
5,2020,Severe,1.00,41.76,7.63,-35.11
6,2021,Recovery,1.00,70.68,4.91,69.25
7,2022,Mild,1.00,100.78,7.85,42.59
8,2023,Moderate,1.00,82.47,8.53,-18.17
9,2024,Moderate,0.58,80.53,8.08,-2.36


In [15]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('KNN Regime by Year', 'KNN Decision Space (Price YoY vs. Balance)'),
                    horizontal_spacing=0.1)

# ── Left: timeline coloured by regime ────────────────────────────────────────
for regime, grp in decomp.groupby('knn_label'):
    fig.add_trace(go.Bar(
        x=grp['year'], y=grp['knn_prob'] * 100,
        name=regime,
        marker_color=REGIME_COLORS.get(regime, '#aaa'),
        hovertemplate='%{x}: <b>' + regime + '</b><br>Confidence: %{y:.0f}%<extra></extra>',
    ), row=1, col=1)

# ── Right: 2D decision space ──────────────────────────────────────────────────
# Background mesh
px_range = np.linspace(decomp['price_yoy'].min()-10, decomp['price_yoy'].max()+10, 60)
pb_range = np.linspace(decomp['balance_mbd'].min()-0.5, decomp['balance_mbd'].max()+0.5, 60)
xx, yy = np.meshgrid(px_range, pb_range)
avg_p3 = decomp['price_vs_3yr'].mean()
mesh_X = np.c_[xx.ravel(), yy.ravel(), np.full(xx.size, avg_p3)]
mesh_Z = knn.predict(scaler.transform(mesh_X)).reshape(xx.shape)

fig.add_trace(go.Heatmap(
    x=px_range, y=pb_range, z=mesh_Z,
    colorscale=[[0,'rgba(255,215,0,0.18)'],[0.33,'rgba(255,152,0,0.18)'],
                [0.67,'rgba(239,83,80,0.18)'],[1,'rgba(76,175,80,0.18)']],
    showscale=False, hoverinfo='skip',
), row=1, col=2)

# Actual data points
for _, row in decomp.iterrows():
    if pd.isna(row['price_yoy']): continue
    fig.add_trace(go.Scatter(
        x=[row['price_yoy']], y=[row['balance_mbd']],
        mode='markers+text',
        text=[str(int(row['year']))],
        textposition='top center',
        marker=dict(color=REGIME_COLORS.get(row['knn_label'],'#aaa'), size=9,
                    line=dict(color='white', width=0.5)),
        showlegend=False,
        hovertemplate=f"<b>{int(row['year'])}</b> — {row['knn_label']}<br>Price YoY: %{{x:.1f}}%<br>Balance: %{{y:+.2f}} Mb/d<extra></extra>",
    ), row=1, col=2)

fig.add_hline(y=0, line=dict(color='#555', dash='dot'), row=1, col=2)
fig.add_vline(x=0, line=dict(color='#555', dash='dot'), row=1, col=2)

fig.update_layout(**DARK, height=480, barmode='stack',
                  title='Method 4 — KNN Regime Classification')
fig.update_xaxes(**AXIS)
fig.update_yaxes(**AXIS)
fig.update_xaxes(title_text='Year', row=1, col=1)
fig.update_yaxes(title_text='Confidence %', row=1, col=1)
fig.update_xaxes(title_text='Price YoY %', row=1, col=2)
fig.update_yaxes(title_text='S-D Balance (Mb/d)', row=1, col=2)
fig.show()

## 6  Composite Demand Destruction Score (DDS)

Combines all four methods into a single −100 → +100 score:

$$\text{DDS} = 0.40 \times \text{EIA signal} + 0.25 \times \text{Trend gap} + 0.20 \times \text{Balance signal} + 0.15 \times \text{Regime penalty}$$

- **< −30**: Bearish demand adjustment warranted in scenario targets
- **−30 to 0**: Mild weakness — monitor
- **> 0**: Demand recovery

In [17]:

def norm(series, low, high):
    """Rescale so that [low, high] maps to [-100, +100]."""
    return np.clip((series - low) / (high - low) * 200 - 100, -100, 100)

dds = decomp[decomp['year'] >= 2015].copy()

# Re-merge counterfactual gap
dds = dds.merge(cf[['year','destruction_kbd']], on='year', how='left')

# Component 1: demand YoY (negative = destruction → positive DDS score)
dds['c1_eia'] = norm(-dds['demand_yoy_d'], -10, 10)

# Component 2: trend gap (positive gap = demand below trend = destruction)
dds['c2_trend'] = norm(dds['destruction_kbd'].fillna(0), -2000, 9000)

# Component 3: supply–demand balance (surplus = potential destruction signal)
dds['c3_balance'] = norm(dds['balance_mbd'], -3, 10)

# Component 4: regime penalty
REGIME_PENALTY = {'Severe': 100, 'Moderate': 50, 'Mild': 10, 'Recovery': -60}
dds['c4_regime'] = dds['knn_label'].map(REGIME_PENALTY).fillna(0)

# Composite
dds['DDS'] = (
    0.40 * dds['c1_eia'] +
    0.25 * dds['c2_trend'] +
    0.20 * dds['c3_balance'] +
    0.15 * dds['c4_regime']
)

current_dds = dds.iloc[-1]
print(f'\n── Composite DDS — {int(current_dds["year"])} ─────────────────────────────')
print(f'  C1 EIA demand signal:   {current_dds["c1_eia"]:+.1f}')
print(f'  C2 Trend gap:           {current_dds["c2_trend"]:+.1f}')
print(f'  C3 Balance:             {current_dds["c3_balance"]:+.1f}')
print(f'  C4 Regime penalty:      {current_dds["c4_regime"]:+.1f}')
print(f'  ── Composite DDS:       {current_dds["DDS"]:+.1f}  (threshold < -30 = bearish)')
dds[['year','DDS','knn_label','brent','balance_mbd','demand_yoy_d']].round(1)



── Composite DDS — 2024 ─────────────────────────────
  C1 EIA demand signal:   -14.3
  C2 Trend gap:           -2.4
  C3 Balance:             +70.5
  C4 Regime penalty:      +50.0
  ── Composite DDS:       +15.3  (threshold < -30 = bearish)


,year,DDS,knn_label,brent,balance_mbd,demand_yoy_d
0,2015,NaN,Recovery,52.4,5.0,NaN
1,2016,-24.4,Recovery,43.5,5.1,1.3
2,2017,-34.2,Recovery,54.2,4.6,2.5
3,2018,-5.6,Mild,71.1,6.9,0.4
4,2019,-19.4,Recovery,64.4,6.1,0.9
5,2020,83.9,Severe,41.8,7.6,-8.9
6,2021,-26.5,Recovery,70.7,4.9,6.5
7,2022,7.4,Mild,100.8,7.8,2.2
8,2023,16.4,Moderate,82.5,8.5,1.7
9,2024,15.3,Moderate,80.5,8.1,1.4


In [18]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.6, 0.4], vertical_spacing=0.06,
                    subplot_titles=('Composite Demand Destruction Score (DDS)', 'Brent Price ($/bbl)'))

# DDS line
dds_colors = ['#EF5350' if v < -30 else '#FFB300' if v < 0 else '#4CAF50' for v in dds['DDS']]
fig.add_trace(go.Bar(
    x=dds['year'], y=dds['DDS'],
    marker_color=dds_colors, name='DDS',
    hovertemplate='%{x}: DDS = <b>%{y:+.1f}</b><extra></extra>',
), row=1, col=1)
fig.add_hline(y=-30, line=dict(color='#EF5350', dash='dash', width=1.5),
              annotation_text='Bearish threshold (−30)', annotation_position='top right',
              row=1, col=1)
fig.add_hline(y=0, line=dict(color='#555', dash='dot'), row=1, col=1)

# Component waterfall (stacked area)
for col_, name_, color_ in [
    ('c1_eia',    'EIA Demand YoY (40%)', '#4FC3F7'),
    ('c2_trend',  'Trend Gap (25%)',       '#FFB300'),
    ('c3_balance','Balance (20%)',         '#7E57C2'),
    ('c4_regime', 'Regime (15%)',          '#EF5350'),
]:
    w = {'c1_eia': 0.40, 'c2_trend': 0.25, 'c3_balance': 0.20, 'c4_regime': 0.15}[col_]
    fig.add_trace(go.Scatter(
        x=dds['year'], y=dds[col_] * w,
        mode='lines', name=name_,
        line=dict(color=color_, width=1.5, dash='dot'),
        hovertemplate=f'{name_}: %{{y:+.1f}}<extra></extra>',
    ), row=1, col=1)

# Brent price
fig.add_trace(go.Scatter(
    x=dds['year'], y=dds['brent'],
    mode='lines+markers', name='Brent $/bbl',
    line=dict(color='#FFB300', width=2),
    hovertemplate='%{x}: <b>$%{y:.0f}/bbl</b><extra>Brent</extra>',
), row=2, col=1)

fig.update_layout(**DARK, height=580, title='Composite Demand Destruction Score vs. Brent Price')
fig.update_xaxes(**AXIS)
fig.update_yaxes(**AXIS)
fig.update_yaxes(title_text='DDS Score', row=1, col=1)
fig.update_yaxes(title_text='$/bbl', row=2, col=1)
fig.show()

## 7  Scenario Sensitivity — Price Impact of Demand Destruction

Using the fitted elasticity ε, estimate how much Brent would need to correct if global demand falls by X Mb/d.  
Uses a simple inverse-elasticity formula: $\Delta P\% = \Delta D\% / \varepsilon$

In [19]:

current_price = combined['brent'].iloc[-1]
base_demand   = combined['global_kbd'].iloc[-1]

# Destruction scenarios (kbd)
scenarios = {
    'Mild (−0.5 Mb/d)':      -500,
    'Moderate (−1.5 Mb/d)':  -1500,
    'Severe (−3.0 Mb/d)':    -3000,
    'COVID-like (−9 Mb/d)':  -9000,
}

rows_s = []
for label, delta_kbd in scenarios.items():
    delta_pct  = delta_kbd / base_demand * 100
    # Δln(P) = Δln(D) / ε  — using IEA consensus ε
    delta_p_pct = delta_pct / IEA_ELASTICITY if IEA_ELASTICITY != 0 else 0
    new_price   = current_price * (1 + delta_p_pct / 100)
    rows_s.append({
        'Scenario': label,
        'Demand Δ (kbd)': delta_kbd,
        'Demand Δ %': f'{delta_pct:.2f}%',
        'Implied Price Δ %': f'{delta_p_pct:+.1f}%',
        'Implied Brent ($/bbl)': round(new_price, 1),
        '_price': new_price,
    })

scen_df = pd.DataFrame(rows_s)
print(f'Using IEA ε = {IEA_ELASTICITY}  |  Current Brent: ${current_price:.1f}/bbl  |  Base demand: {base_demand:,.0f} kbd\n')
print(scen_df[['Scenario','Demand Δ (kbd)','Demand Δ %','Implied Price Δ %','Implied Brent ($/bbl)']].to_string(index=False))


Using IEA ε = -0.06  |  Current Brent: $80.5/bbl  |  Base demand: 78,716 kbd

            Scenario  Demand Δ (kbd) Demand Δ % Implied Price Δ %  Implied Brent ($/bbl)
    Mild (−0.5 Mb/d)            -500     -0.64%            +10.6%                   89.1
Moderate (−1.5 Mb/d)           -1500     -1.91%            +31.8%                  106.1
  Severe (−3.0 Mb/d)           -3000     -3.81%            +63.5%                  131.7
COVID-like (−9 Mb/d)           -9000    -11.43%           +190.6%                  234.0


In [20]:

# Sensitivity curve: demand destruction -10 Mb/d to +3 Mb/d
d_range_kbd = np.linspace(-10000, 3000, 200)
d_pct       = d_range_kbd / base_demand * 100
p_pct       = d_pct / IEA_ELASTICITY
p_implied   = current_price * (1 + p_pct / 100)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=d_range_kbd / 1000, y=p_implied,
    mode='lines', name=f'Price sensitivity (IEA ε={IEA_ELASTICITY})',
    line=dict(color='#4FC3F7', width=2.5),
    hovertemplate='Demand Δ: %{x:.1f} Mb/d<br>Implied Brent: <b>$%{y:.1f}/bbl</b><extra></extra>',
))

for row in rows_s:
    fig.add_trace(go.Scatter(
        x=[row['Demand Δ (kbd)'] / 1000], y=[row['_price']],
        mode='markers+text',
        text=[row['Scenario'].split('(')[0].strip()],
        textposition='top center',
        marker=dict(size=12, color='#FFB300', line=dict(color='white', width=1)),
        showlegend=False,
        hovertemplate=f"<b>{row['Scenario']}</b><br>Implied Brent: ${row['_price']:.1f}/bbl<extra></extra>",
    ))

fig.add_hline(y=current_price, line=dict(color='#aaa', dash='dot'),
              annotation_text=f'Current Brent ${current_price:.0f}/bbl')
fig.add_vline(x=0, line=dict(color='#555', dash='dot'))

fig.add_hrect(y0=0,   y1=50,  fillcolor='rgba(239,83,80,0.08)',  line_width=0, annotation_text='Shale break-even zone')
fig.add_hrect(y0=100, y1=220, fillcolor='rgba(255,179,0,0.08)', line_width=0, annotation_text='Demand destruction zone')

fig.update_layout(
    **DARK, height=500,
    title=f'Scenario Sensitivity — Implied Brent vs. Demand Destruction (IEA ε = {IEA_ELASTICITY})',
    xaxis=dict(**AXIS, title='Global Demand Change (Mb/d)'),
    yaxis=dict(**AXIS, title='Implied Brent ($/bbl)'),
    showlegend=False,
)
fig.show()


## 8  Country-Level Demand Breakdown — Who Drives the Destruction?

Which of the 25 tracked countries account for the most demand contraction in each episode? Uses the EIA annual data already in the `supply_demand` DB table.

In [21]:
from config import SD_COUNTRIES

# Pivot demand by country × year
pivot = demand_db.pivot_table(index='year', columns='country', values='kbd', aggfunc='sum')
yoy_pct  = pivot.pct_change() * 100
yoy_kbd  = pivot.diff()

# ── Heatmap: YoY % demand change ─────────────────────────────────────────────
countries_sorted = pivot.mean().sort_values(ascending=False).index.tolist()
labels = [SD_COUNTRIES.get(c, c) for c in countries_sorted]

fig = go.Figure(go.Heatmap(
    x=yoy_pct.index.tolist(),
    y=labels,
    z=yoy_pct[countries_sorted].values,
    colorscale=[
        [0.0, '#C0392B'], [0.35, '#7B241C'],
        [0.5, '#2c2c42'],
        [0.65, '#1A5276'], [1.0, '#2ECC71'],
    ],
    zmid=0, zmin=-15, zmax=15,
    colorbar=dict(title='YoY %', tickfont=dict(color='#B0B0C0', size=10)),
    hovertemplate='%{y} — %{x}: <b>%{z:+.1f}%</b><extra></extra>',
))

# Highlight known destruction years
for yr in [2009, 2020]:
    fig.add_vline(x=yr, line=dict(color='#EF5350', dash='dash', width=1.5))

fig.update_layout(
    **DARK, height=600,
    title='Country-Level Demand YoY % Change — Demand Destruction Episodes Highlighted',
    xaxis=dict(**AXIS, title='Year'),
    yaxis=dict(**AXIS, autorange='reversed'),
)
fig.show()

In [22]:
# ── Waterfall: top contributors to demand change in worst episode (2020) ─────
episode_yr = 2020
if episode_yr in yoy_kbd.index:
    changes = yoy_kbd.loc[episode_yr].dropna().sort_values()
    changes_labeled = pd.Series(
        changes.values, index=[SD_COUNTRIES.get(c, c) for c in changes.index]
    )

    fig = go.Figure(go.Waterfall(
        x=changes_labeled.index.tolist(),
        y=changes_labeled.values,
        measure=['relative'] * len(changes_labeled),
        decreasing=dict(marker_color='#EF5350'),
        increasing=dict(marker_color='#4CAF50'),
        totals=dict(marker_color='#4FC3F7'),
        connector=dict(line=dict(color='#2c2c42', width=1)),
        hovertemplate='<b>%{x}</b>: %{y:+,.0f} kbd<extra></extra>',
    ))

    fig.update_layout(
        **DARK, height=420,
        title=f'Demand Change by Country — {episode_yr} (kbd YoY)',
        xaxis=dict(**AXIS, title='Country', tickangle=-40),
        yaxis=dict(**AXIS, title='kbd YoY change'),
    )
    fig.show()
else:
    print(f'Year {episode_yr} not in dataset — adjust episode_yr variable')

## 9  Summary — Key Calibration Numbers

These are the model outputs to carry into scenario analysis and the Composite DDS formula.

In [23]:

current_row = dds.iloc[-1]

print('═' * 62)
print('  DEMAND DESTRUCTION MODEL — CALIBRATION SUMMARY')
print('═' * 62)
print(f'  Reference date:          {int(current_row["year"])}')
print(f'  Brent price:             ${current_row["brent"]:.1f} / bbl')
print()
print(f'  ── Method 1: Price Elasticity ────────────────────────')
print(f'     OLS β (activity cycle): {slope:+.4f}  R²={r**2:.2f}')
print(f'     IEA consensus ε:        {IEA_ELASTICITY:.2f} (used in scenarios)')
print(f'     At $150/bbl (+50%):     {IEA_ELASTICITY*50:.1f}% demand change')
delta = GLOBAL_BASE_KBD * IEA_ELASTICITY * 50 / 100
print(f'     = {delta:+,.0f} kbd  ({delta/1000:+.2f} Mb/d)')
print()
print(f'  ── Method 2: Counterfactual Trend ────────────────────')
print(f'     Trend growth:         +{slope_t:,.0f} kbd/yr')
cf_latest = cf[cf['year'] == int(current_row['year'])]
if not cf_latest.empty:
    gap = cf_latest['destruction_kbd'].values[0]
    print(f'     Current gap vs trend: {gap:+,.0f} kbd ({gap/1000:+.2f} Mb/d)')
print()
print(f'  ── Method 3: Balance ─────────────────────────────────')
print(f'     S-D balance:          {current_row["balance_mbd"]:+.2f} Mb/d')
print(f'     Regime (balance):     {current_row["regime"]}')
print()
print(f'  ── Method 4: KNN Regime ──────────────────────────────')
print(f'     KNN regime:           {current_row["knn_label"]}  ({current_row["knn_prob"]*100:.0f}% confidence)')
print()
print(f'  ── Composite DDS ─────────────────────────────────────')
dds_val = current_row['DDS']
verdict = 'BEARISH — demand adjustment warranted' if dds_val < -30 else \
          'WEAK — monitor closely' if dds_val < 0 else 'NEUTRAL / RECOVERY'
print(f'     DDS:                  {dds_val:+.1f}  →  {verdict}')
print('═' * 62)


══════════════════════════════════════════════════════════════
  DEMAND DESTRUCTION MODEL — CALIBRATION SUMMARY
══════════════════════════════════════════════════════════════
  Reference date:          2024
  Brent price:             $80.5 / bbl

  ── Method 1: Price Elasticity ────────────────────────
     OLS β (activity cycle): +0.1098  R²=0.62
     IEA consensus ε:        -0.06 (used in scenarios)
     At $150/bbl (+50%):     -3.0% demand change
     = -2,361 kbd  (-2.36 Mb/d)

  ── Method 2: Counterfactual Trend ────────────────────
     Trend growth:         +974 kbd/yr
     Current gap vs trend: +3,366 kbd (+3.37 Mb/d)

  ── Method 3: Balance ─────────────────────────────────
     S-D balance:          +8.08 Mb/d
     Regime (balance):     Demand Destruction

  ── Method 4: KNN Regime ──────────────────────────────
     KNN regime:           Moderate  (58% confidence)

  ── Composite DDS ─────────────────────────────────────
     DDS:                  +15.3  →  NEUTRAL / RECOVER